In [ ]:
!pip install langchain faiss-cpu langchain-community langchain-huggingface sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

/tmp/ipykernel_7440/2172280237.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Documents

In [3]:
doc1 = Document(
    page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
    metadata={"team": "Royal Challengers Bangalore"}
)
doc2 = Document(
    page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
    metadata={"team": "Mumbai Indians"}
)
doc3 = Document(
    page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
    metadata={"team": "Chennai Super Kings"}
)
doc4 = Document(
    page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
    metadata={"team": "Mumbai Indians"}
)
doc5 = Document(
    page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
    metadata={"team": "Chennai Super Kings"}
)

docs = [doc1, doc2, doc3, doc4, doc5]

Init vector store

In [4]:
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# FAISS builds the index directly from documents — no separate add_documents needed
vector_store = FAISS.from_documents(docs, embedding)

# Unlike Chroma where you init first then add_documents, FAISS builds the index in one shot via from_documents.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Add more documents later

In [ ]:
# if you want to add docs after creation
vector_store.add_documents([doc1])

 Similarity search

In [5]:
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='95a6d849-e449-41be-9ef6-35b2c52ea942', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='a0a162ec-dd04-4a28-9c33-9df1378bd9e6', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

Similarity search with score

FAISS returns L2 distance (lower = more similar), unlike Chroma's cosine distance. So scores here will look different numerically but the ranking is what matters.

In [6]:
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='95a6d849-e449-41be-9ef6-35b2c52ea942', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  np.float32(0.9693598)),
 (Document(id='a0a162ec-dd04-4a28-9c33-9df1378bd9e6', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  np.float32(1.1493447))]

Metadata filtering

In [7]:
vector_store.similarity_search(
    query='good captain',
    k=2,
    filter={"team": "Chennai Super Kings"}
)

[Document(id='f1074309-381a-4350-b4db-95d0c4c1d99a', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
 Document(id='2dab04f4-f518-4574-9e8a-5606840b595d', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

 Persist to disk

In [8]:
vector_store.save_local("faiss_index")

Reload from disk

In [9]:
vector_store = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

Delete document

In [10]:
# FAISS delete needs the internal docstore id, not the metadata id
# first find it via similarity search and inspect
results = vector_store.similarity_search("Virat Kohli", k=1)

# docstore ids are stored in index_to_docstore_id
print(vector_store.index_to_docstore_id)  # {0: 'abc123', 1: 'def456', ...}

{0: '64dfdbfb-b61f-4af6-97d9-375391c65630', 1: 'a0a162ec-dd04-4a28-9c33-9df1378bd9e6', 2: 'f1074309-381a-4350-b4db-95d0c4c1d99a', 3: '95a6d849-e449-41be-9ef6-35b2c52ea942', 4: '2dab04f4-f518-4574-9e8a-5606840b595d'}


In [11]:
# then delete by that id
vector_store.delete(['64dfdbfb-b61f-4af6-97d9-375391c65630'])

True

In [12]:
# view all documents currently in the store
for doc_id, doc in vector_store.docstore._dict.items():
    print(doc_id)
    print(doc.page_content[:80])
    print(doc.metadata)
    print("---")

a0a162ec-dd04-4a28-9c33-9df1378bd9e6
Rohit Sharma is the most successful captain in IPL history, leading Mumbai India
{'team': 'Mumbai Indians'}
---
f1074309-381a-4350-b4db-95d0c4c1d99a
MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multipl
{'team': 'Chennai Super Kings'}
---
95a6d849-e449-41be-9ef6-35b2c52ea942
Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playin
{'team': 'Mumbai Indians'}
---
2dab04f4-f518-4574-9e8a-5606840b595d
Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball.
{'team': 'Chennai Super Kings'}
---
